# Worspace price tier classification model

Predicting price tier of a work space based on its features set.<br>
Exploring whole machine learning pipeline for making usefull solution along the way.

### **VERSION v.0.1**
+ Excluding top null rankers from preprocessing pipeline
+ Using Maximum features with mid tollerance for null values being loaded
+ encoding pipeline with switches for bench marking outputs
+ step by step function loading based benchmark setup
+ global chain of config flow
+ simple preprocessing => filter => selection => encoding pipeline


In [9]:
# GLOBAL CONFIG
from sklearn import set_config

TRAIN_DATASET = '/home/madhav/workstation/Lab/experiment/data/MLP-PROJECT-DATASET/train.csv'
TEST_DATASET = '/home/madhav/workstation/Lab/experiment/data/MLP-PROJECT-DATASET/test.csv'

EXCLUSION_INDEX = [
    'operator_has_avatar',
]
TARGET = 'price_tier'
ID = 'space_id'

# Encoding Loading
one_hot_targets = [
    'operator_id_verified',
    'operator_certified',
    'compliance_status',
    'bookable'
]
target_targets = [
    'operator_class',
    'zone',
    'operator_ref'
]

# GLOBAL CONFIG FOR SCIKIT LEARN
set_config(transform_output="pandas")

In [10]:
# Data loading

import pandas as pd
from rich import print

raw_df = pd.read_csv(TRAIN_DATASET)


# Foundational Data cleaning sequence
'''
Data Inspection pattern
with sequential null values with scores along with review date pattern,
iterative models would first test out with filling those values with zero, 
dropping borough due to non-patternic null values ranges
'''
raw_df.drop(columns=['borough'], inplace=True) # removing largest null ranker
raw_df.reset_index(drop=True, inplace=True)
ranking = raw_df.isna().sum().sort_values(ascending=False)

# Null values inspection
inspection_required = False
if inspection_required:
    for _ in ranking.index:
        print(f'''
            {'-'*10} \n 
            Sample insepction of {_} \n
            {raw_df[_].sample(10)} \n
            Total Null values [red] {ranking[_]} [/red]
        ''')

'''
--- Null Inspection report ---
54240 Null values form review start and end dates
Taking cascading effect to plenty of features,
Thus being a big number to handle the null we might need to inspect for distribution
and pattern to deal with these null patterns
'''

NULL_RANKERS = ranking[:11].index

In [11]:
raw_df.columns

Index(['space_id', 'operator_ref', 'operator_platform_age_years',
       'operator_platform_age_months', 'operator_active_years',
       'operator_active_months', 'operator_class', 'operator_certified',
       'operator_portfolio_size', 'operator_has_avatar',
       'operator_id_verified', 'zone', 'geo_lat', 'geo_lon', 'space_class',
       'access_type', 'max_seats', 'utility_rooms', 'zones', 'workstations',
       'facilities', 'snapshot_start_date', 'snapshot_end_date',
       'min_term_days', 'max_term_days', 'min_term_floor', 'min_term_ceiling',
       'max_term_floor', 'max_term_ceiling', 'min_term_avg', 'max_term_avg',
       'bookable', 'open_slots_a', 'open_slots_b', 'open_slots_c',
       'open_slots_d', 'review_count', 'review_count_12m', 'review_count_30d',
       'open_slots_e', 'review_count_prev_yr', 'booked_load',
       'first_review_date', 'last_review_date', 'score_overall',
       'score_accuracy', 'score_condition', 'score_onboarding',
       'score_support', 'scor

### Data preprocessing

Critical data leakage learning, 
Imputing before spliting dataset into train, test, and validation set, 
commits data leakage, thus first step is to split the dataset, 
fit_transform imputers and transformers into training set, and try out with validation with testing setup, 

Initial preprocessing unit taking simple transformation with columns, 
doing median and most frequent based transformation to handle null values

In [12]:
# Initiating Data set split
from sklearn.model_selection import train_test_split

# Loading Dataset
'''
Loader for testing and training loader
Target handle with training switch
test switch, 

Loading just the required columns into the main load 
due to abundance of too much non-required features lying along with
versioning feature loading into main processing frame

TOTAL NUMBER OF FEATURE FOR LOAD : 46
'''

# Tested
def load_dataset(raw_df:pd.DataFrame, training=True):
    '''
    Initial data set loading utility
    Index load targeted features into gloabal config cell
    pass training=True for target extraction, 
    or leave for test set extraction data frame

    return : data frame with features
        target vector only returned with training parameter call
    '''
    # Foundational removal of explicit null rankers
    features = raw_df.drop(columns=EXCLUSION_INDEX).copy(deep=True)
    features.drop(columns=NULL_RANKERS, inplace=True)
    features.reset_index(drop=True, inplace=True)

    features.drop(columns=ID, inplace=True)
    features.reset_index(drop=True, inplace=True)

    if not training:
        ids = raw_df[ID] # anchor id
        return features, ids
    features = features.drop(columns=['price_tier'])
    target = raw_df[TARGET]
    # removing id feature for training dataset
    return features, target

# Training Load
X, y = load_dataset(raw_df, training=True)  # Fetchout for trian set

In [13]:
# Data Disection
from sklearn.model_selection import train_test_split

X_val_train, X_test, y_val_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_val_train, y_val_train, test_size=0.20, random_state=42
)

print(f"""
    Data disect inspection \n 
    training : {X_train.shape, y_train.shape} \n
    testing : {X_test.shape, y_test.shape} \n 
    Validation : {X_val.shape, y_val.shape}
""")

Data disect inspection 
 
    training : ((179036, 45), (179036,)) 

    testing : ((55950, 45), (55950,)) 
 
    Validation : ((44760, 45), (44760,))

### Pipeline construction

Building preprocessing pipeline with transformers and pipeline, pipeline executes the transformation and imputation in sequential format, whereas `ColumnTransformer` makes paralell execution of all process, 

Using transformer to assemble multiple pipelines, making efficient preprocessing unit for initial load sequence

### Internals of preprocessing pipeline

Initial foundational data filteration and cleaning, removing null values,
and most essential imputations for further encoding requriements setup.

For referencing targets refer global indexing cell
+ Numerical with median imputations
+ Categorical Null dealing witnh most frequent replacement

----

With foundational transformation with imputing median and most frequent, dataset is cleaned for taking account of foundational models for bench marking scores and improvement directions with neew upgrades to encoders and new set of features introduction.


In [14]:
from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer, make_column_selector

'''
Preprocessing layer

Numerical imputer : Median
Categorical imputer : most frequent
'''


numerical_columns = [ col for col in X_train.columns if (X_train[col].dtype in ['float64', 'int64'])]
pre_transformer = ColumnTransformer(
    [
        # Imputation target with all numerical points
        ('numeric_imputer',
        SimpleImputer(strategy='median'),
        numerical_columns),
        
        # Imputing Most frequent to all categorical dataset
        ('categorical imputer',
        SimpleImputer(strategy='most_frequent'),
        make_column_selector(dtype_include=['object', 'category']))
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
    
)


## Transformer prime

Transforming categorical into numerical, along with standardizing with scaling, binning and polynomial feature addition to the original dataset

With transformation we would be converting raw dataset into numerical features to start bench marking with foundational models and start the training and model benchmarking with the dataset

### Categorical dataset encoding
1. OneHotEncoding
    - operator_id_verified
    - operator_certified
    - compliance_status
    - bookable

2. TargetEncoding
    - operator_class
    - zone
    - operator_ref

3. Frequency based
    - space_class
    - access_type
    - metro

4. MultiLabelBinarizer
    - facilities

### Numerical Extractions and relations

1. Geo-location decomposition
    - converting location to pca 1
2. Years + month = date => standard scaling

In [15]:
# ENCODING
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, MultiLabelBinarizer, MinMaxScaler
from sklearn.pipeline import Pipeline


ignition_sequence = ColumnTransformer([
    (
        'categorical encoding',
        OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False),
        one_hot_targets
    ),
    (
        'target encoding',
        TargetEncoder(target_type='continuous'),
        target_targets
    )
])



# Transforming testing validation and training dataset

def ignite(feats, target):
    ''' 
    For training sequence only with
    fit_transformed api endpoint
    Making simple loading sequence for the
    ignition sequence for the dataset loading
    '''
    df_unit = pd.DataFrame()
    preprocessed = pre_transformer.fit_transform(feats, target)
    df_unit = pd.concat([df_unit, preprocessed], ignore_index=True)

    transformed = ignition_sequence.fit_transform(feats, target)
    df_unit = pd.concat([df_unit, transformed], ignore_index=True)
    return df_unit

def thrust(df):
    '''
    For testing and validation dataset
    just transforming with original dataset
    '''
    df_unit = pd.DataFrame()
    preprocessed = pre_transformer.transform(df)
    df_unit = pd.concat([df_unit, preprocessed])

    transformed = ignition_sequence.transform(df)
    df_unit = pd.concat([df_unit, transformed])
    return df_unit


# Initial Preprocessing
X_train = ignite(X_train, y_train)
X_test = thrust(X_test)
X_val = thrust(X_val)

print(X_train.isna().sum())
X_train.head()

operator_platform_age_years                           179036
operator_platform_age_months                          179036
operator_active_years                                 179036
operator_active_months                                179036
operator_portfolio_size                               179036
geo_lat                                               179036
geo_lon                                               179036
max_seats                                             179036
utility_rooms                                         179036
workstations                                          179036
min_term_days                                         179036
max_term_days                                         179036
min_term_floor                                        179036
min_term_ceiling                                      179036
max_term_floor                                        179036
max_term_ceiling                                      179036
min_term_avg                                          179036
max_term_avg                                          179036
open_slots_a                                          179036
open_slots_b                                          179036
open_slots_c                                          179036
open_slots_d                                          179036
review_count                                          179036
review_count_12m                                      179036
review_count_30d                                      179036
open_slots_e                                          179036
review_count_prev_yr                                  179036
booked_load                                           179036
operator_metro_units                                  179036
operator_metro_whole                                  179036
operator_metro_private                                179036
operator_metro_shared                                 179036
operator_ref                                          179036
operator_class                                        179036
operator_certified                                    179036
operator_id_verified                                  179036
zone                                                  179036
space_class                                           179036
access_type                                           179036
facilities                                            179036
snapshot_start_date                                   179036
snapshot_end_date                                     179036
bookable                                              179036
compliance_status                                     179036
metro                                                 179036
categorical encoding__operator_id_verified_t          179036
categorical encoding__operator_id_verified_nan        179036
categorical encoding__operator_certified_t            179036
categorical encoding__operator_certified_nan          179036
categorical encoding__compliance_status_in_review     179036
categorical encoding__compliance_status_licensed      179036
categorical encoding__compliance_status_unlicensed    179036
categorical encoding__bookable_t                      179036
categorical encoding__bookable_nan                    179036
target encoding__operator_class                       179036
target encoding__zone                                 179036
target encoding__operator_ref                         179036
dtype: int64

,operator_platform_age_years,operator_platform_age_months,operator_active_years,operator_active_months,operator_portfolio_size,geo_lat,geo_lon,max_seats,utility_rooms,workstations,...,categorical encoding__operator_certified_t,categorical encoding__operator_certified_nan,categorical encoding__compliance_status_in_review,categorical encoding__compliance_status_licensed,categorical encoding__compliance_status_unlicensed,categorical encoding__bookable_t,categorical encoding__bookable_nan,target encoding__operator_class,target encoding__zone,target encoding__operator_ref
0,10.0,0.0,2.0,7.0,2.0,11.0580,-64.6804,2.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6.0,1.0,6.0,1.0,1042.0,96.5670,-33.8938,4.0,2.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5.0,10.0,5.0,10.0,20.0,97.3770,-8.4190,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3.0,11.0,3.0,4.0,1.0,87.8865,-123.2896,8.0,2.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10.0,4.0,10.0,4.0,8.0,10.9770,-64.9300,16.0,2.0,16.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Data set pipeline contruction

We load our features set with targeted, onehot along with standardScaler with numerical features to serve our models,
Testing Load snnipit
```python

df = pd.read_csv(TEST_DATASET)

test_df, id = load_dataset(df, training=False)

processed_df = preprocessing_handle.transform(test_df)
final_df = ignition_encoder.transform(processed_df)
```

In [16]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Initialize and Train the Decision Tree on the Training Set
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

# 2. Use the Validation Set to Tune / Check Performance (Unseen during training)
y_val_pred = clf.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f"Validation Accuracy: {val_accuracy:.4f}")

# 3. Use the Testing Set for the Final Evaluation (Only once your model is finalized)
y_test_pred = clf.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")

# Detailed report for test set
print("\nTest Classification Report:")
print(classification_report(y_test, y_test_pred))

ValueError: could not convert string to float: 'Nalnar'

In [ ]:
# MODEL XGBOOOST
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Map target labels down by 1 so they start at 0 (e.g., 1->0, 2->1, etc.)
y_train_mapped = y_train - 1
y_val_mapped = y_val - 1
y_test_mapped = y_test - 1

# 2. Initialize the XGBoost Classifier
# eval_metric='mlogloss' is standard for multi-class classification
xgb_clf = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=20,
    random_state=42,
    eval_metric='mlogloss',
    n_jobs=-1
)

# 3. Train the model on the mapped training set
xgb_clf.fit(X_train, y_train_mapped)

# 4. Predict and evaluate on the Validation set
y_val_pred_xgb = xgb_clf.predict(X_val)
val_acc_xgb = accuracy_score(y_val_mapped, y_val_pred_xgb)
print(f"XGBoost Validation Accuracy: {val_acc_xgb:.4f}")

# 5. Predict and evaluate on the Test set
y_test_pred_xgb = xgb_clf.predict(X_test)
test_acc_xgb = accuracy_score(y_test_mapped, y_test_pred_xgb)
print(f"XGBoost Test Accuracy: {test_acc_xgb:.4f}\n")

# 6. Detailed Classification Report (Map predictions back to 1-4 for clean reading)
print("XGBoost Test Classification Report:")
print(classification_report(y_test_mapped + 1, y_test_pred_xgb + 1))


XGBoost Validation Accuracy: 0.6764

XGBoost Test Accuracy: 0.6771

XGBoost Test Classification Report:

precision    recall  f1-score   support

           1       0.79      0.78      0.79     13324
           2       0.62      0.67      0.65     16327
           3       0.53      0.51      0.52     12163
           4       0.76      0.73      0.74     14136

    accuracy                           0.68     55950
   macro avg       0.68      0.67      0.67     55950
weighted avg       0.68      0.68      0.68     55950